In [1]:
from unc_handling import UG_prompter
from DataLoader import DataLoader
from segmentation import Segmentation
from segmentation_util import combine_prompt_sets
from evaluation import Evaluator, compare_to_recontours

root = r"C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\data\LUNDPROBE\ExtendedSamples\development"
methods_available = ["raycast", "local_normals"]
method = methods_available[1]
run_eval = True
evaluate_compare_recontours = True

data = DataLoader(parentfolder=root,subject_nr=1,volume_of_interest="CTVT",verbose=True)
unc_handler = UG_prompter(data=data)
seg_handler = Segmentation(data=data)

Loaded subject newAcq_0b4940fa31a1d650 with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024), Ground truth shape: (88, 1024, 1024)
Image spacing (z, y, x): (2.5, 0.46880000829696655, 0.46880000829696655)
Initilialized UG_prompter for subject 1 with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024).
Initialized Segmentation for subject 1 with volume of interest 'CTVT'
Mask shape: (88, 1024, 1024)
Building SAM predictor from checkpoint: checkpoints/MedSAM2_latest.pt


c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\modeling\sam\transformer.py:23: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()


In [2]:
# ============================================================
# Interactive slice viewer: MRI + uncertainty overlay + mask contour
# Includes:
# - slice slider
# - alpha slider
# - uncertainty threshold slider
# - uncertainty percentile clipping slider
# - zoom slider
# - save button for current selected slice
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
from skimage.measure import find_contours


def interactive_uncertainty_viewer(
    img,
    unc_map,
    mask=None,
    figsize=(7, 7),
    cmap_unc="turbo",          # green/blue/red-ish; use "jet" if you prefer classic blue-green-red
    alpha_init=0.45,
    percentile_clip_init=99,
    zoom_fraction_init=0.30,
    output_folder="uncertainty_slice_exports",
    mask_color="orange",
    mask_linewidth=1.4,
):
    img = np.asarray(img)
    unc_map = np.asarray(unc_map)

    if img.shape != unc_map.shape:
        raise ValueError(f"Shape mismatch: img={img.shape}, unc_map={unc_map.shape}")

    if mask is not None:
        mask = np.asarray(mask).astype(bool)
        if mask.shape != img.shape:
            raise ValueError(f"Shape mismatch: mask={mask.shape}, img={img.shape}")

    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    n_slices = img.shape[0]

    valid_unc = unc_map[np.isfinite(unc_map)]
    if valid_unc.size == 0:
        raise ValueError("unc_map contains no finite values.")

    unc_min = float(np.nanmin(valid_unc))
    unc_max = float(np.nanmax(valid_unc))
    unc_step = (unc_max - unc_min) / 300 if unc_max > unc_min else 1.0

    slice_slider = widgets.IntSlider(
        value=n_slices // 2,
        min=0,
        max=n_slices - 1,
        step=1,
        description="slice",
        continuous_update=False,
    )

    alpha_slider = widgets.FloatSlider(
        value=alpha_init,
        min=0.0,
        max=1.0,
        step=0.05,
        description="alpha",
        continuous_update=False,
    )

    clip_slider = widgets.FloatSlider(
        value=percentile_clip_init,
        min=80,
        max=100,
        step=0.5,
        description="clip %",
        continuous_update=False,
    )

    threshold_slider = widgets.FloatSlider(
        value=unc_min,
        min=unc_min,
        max=unc_max,
        step=unc_step,
        description="threshold",
        continuous_update=False,
    )

    zoom_slider = widgets.FloatSlider(
        value=zoom_fraction_init,
        min=0.05,
        max=1.0,
        step=0.05,
        description="zoom frac",
        continuous_update=False,
    )

    zoom_checkbox = widgets.Checkbox(
        value=False,
        description="zoom",
    )

    show_overlay_checkbox = widgets.Checkbox(
        value=True,
        description="unc overlay",
    )

    show_mask_checkbox = widgets.Checkbox(
        value=True,
        description="mask contour",
    )

    show_colorbar_checkbox = widgets.Checkbox(
        value=True,
        description="colorbar",
    )

    save_button = widgets.Button(
        description="Save current slice",
        button_style="success",
        tooltip="Save the currently displayed slice as PNG",
    )

    save_status = widgets.HTML(value="")

    out = widgets.Output()

    last_render = {
        "fig": None,
        "z": None,
        "y0": None,
        "y1": None,
        "x0": None,
        "x1": None,
    }

    def draw_contour(ax, mask2d, color="cyan", linewidth=1.4):
        if mask2d is None or not np.any(mask2d):
            return

        contours = find_contours(mask2d.astype(float), 0.5)

        for contour in contours:
            ax.plot(
                contour[:, 1],
                contour[:, 0],
                color=color,
                linewidth=linewidth,
            )

    def render_current_slice(savepath=None, show=True):
        z = slice_slider.value
        alpha = alpha_slider.value
        clip_pct = clip_slider.value
        threshold = threshold_slider.value
        zoom = zoom_checkbox.value
        zoom_fraction = zoom_slider.value
        show_overlay = show_overlay_checkbox.value
        show_mask = show_mask_checkbox.value
        show_colorbar = show_colorbar_checkbox.value

        img_z = img[z]
        unc_z = unc_map[z]

        unc_thr = np.where(unc_z >= threshold, unc_z, np.nan)

        vmax = np.nanpercentile(valid_unc, clip_pct)
        vmin = threshold

        if vmax <= vmin:
            vmax = float(np.nanmax(valid_unc))

        H, W = img_z.shape

        fg = np.zeros_like(img_z, dtype=bool)

        if np.any(np.isfinite(unc_thr)):
            fg |= np.isfinite(unc_thr)

        if mask is not None and np.any(mask[z]):
            fg |= mask[z]

        if zoom and fg.any():
            ys, xs = np.where(fg)

            cy = int(np.round(np.mean(ys)))
            cx = int(np.round(np.mean(xs)))

            h_half = int(H * zoom_fraction / 2)
            w_half = int(W * zoom_fraction / 2)

            y0 = max(0, cy - h_half)
            y1 = min(H, cy + h_half)
            x0 = max(0, cx - w_half)
            x1 = min(W, cx + w_half)
        else:
            y0, y1 = 0, H
            x0, x1 = 0, W

        fig, ax = plt.subplots(figsize=figsize)

        ax.imshow(img_z[y0:y1, x0:x1], cmap="gray")

        im_unc = None

        if show_overlay:
            im_unc = ax.imshow(
                unc_thr[y0:y1, x0:x1],
                cmap=cmap_unc,
                alpha=alpha,
                vmin=vmin,
                vmax=vmax,
            )

        if mask is not None and show_mask:
            draw_contour(
                ax,
                mask[z, y0:y1, x0:x1],
                color=mask_color,
                linewidth=mask_linewidth,
            )

        ax.set_title(
            f"Slice {z} | threshold ≥ {threshold:.4g} | clip {clip_pct:g}% | zoom {zoom_fraction:.2f}"
        )
        ax.set_axis_off()

        if show_overlay and show_colorbar and im_unc is not None:
            cbar = plt.colorbar(im_unc, ax=ax, fraction=0.046, pad=0.04)
            cbar.set_label("Uncertainty")

        plt.tight_layout()

        if savepath is not None:
            fig.savefig(savepath, dpi=300, bbox_inches="tight")

        last_render.update({
            "fig": fig,
            "z": z,
            "y0": y0,
            "y1": y1,
            "x0": x0,
            "x1": x1,
        })

        if show:
            plt.show()
        else:
            plt.close(fig)

    def update(*args):
        with out:
            clear_output(wait=True)
            render_current_slice(show=True)

    def save_current_slice(_):
        z = slice_slider.value
        zoom_label = "zoom" if zoom_checkbox.value else "full"

        savepath = output_folder / (
            f"uncertainty_overlay_slice_{z:03d}_{zoom_label}.png"
        )

        render_current_slice(savepath=savepath, show=False)

        save_status.value = f"<b>Saved:</b> {savepath}"

    for w in [
        slice_slider,
        alpha_slider,
        clip_slider,
        threshold_slider,
        zoom_slider,
        zoom_checkbox,
        show_overlay_checkbox,
        show_mask_checkbox,
        show_colorbar_checkbox,
    ]:
        w.observe(update, names="value")

    save_button.on_click(save_current_slice)

    controls = widgets.VBox([
        slice_slider,
        widgets.HBox([alpha_slider, clip_slider, threshold_slider]),
        widgets.HBox([zoom_checkbox, zoom_slider]),
        widgets.HBox([show_overlay_checkbox, show_mask_checkbox, show_colorbar_checkbox]),
        widgets.HBox([save_button, save_status]),
    ])

    display(controls, out)
    update()


# ============================================================
# Call
# ============================================================

interactive_uncertainty_viewer(
    img=data.img,
    unc_map=data.unc_map,
    mask=data.mask,
    figsize=(7, 7),
    cmap_unc="turbo",       # try "jet" if you want the classic blue-green-red
    alpha_init=0.45,
    percentile_clip_init=99,
    zoom_fraction_init=0.30,
    output_folder="uncertainty_slice_exports",
    mask_color="orange",
    mask_linewidth=1.4,
)

Output()

In [25]:
# ============================================================
# GIF: Uncertainty threshold sweep for a single slice
# Safe version: fixes vmin/vmax issue + fixed 20% zoom
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.animation import FuncAnimation, PillowWriter


def make_uncertainty_threshold_sweep_gif(
    img,
    unc_map,
    slice_idx=41,
    thresholds=None,
    savepath="uncertainty_threshold_sweep_slice_041.gif",
    fps=2,
    figsize=(7, 7),
    cmap_unc="turbo",
    alpha=0.55,
    zoom_fraction=0.20,
):

    img = np.asarray(img)
    unc_map = np.asarray(unc_map)

    if img.shape != unc_map.shape:
        raise ValueError(f"Shape mismatch: img={img.shape}, unc_map={unc_map.shape}")

    if thresholds is None:
        thresholds = np.array([
            0.194962, 0.097481, 0.048740, 0.024370, 0.012185,
            0.006093, 0.009139, 0.010662, 0.011424, 0.011804,
            0.011995, 0.012090, 0.012042, 0.012019, 0.012007,
            0.012001, 0.011998, 0.011999, 0.011998, 0.011998,
        ], dtype=float)
    else:
        thresholds = np.asarray(thresholds, dtype=float)

    img_z = img[slice_idx]
    unc_z = unc_map[slice_idx]

    lowest_thr = float(np.min(thresholds))
    highest_thr = float(np.max(thresholds))

    # fixed normalization range that is always valid
    vmin = lowest_thr
    vmax = float(np.nanmax(unc_z))

    if not np.isfinite(vmax) or vmax <= vmin:
        vmax = highest_thr

    if vmax <= vmin:
        vmax = vmin + 1e-8

    # fixed zoom window based on lowest threshold
    fg = unc_z >= lowest_thr

    H, W = img_z.shape

    if np.any(fg):
        ys, xs = np.where(fg)

        cy = int(np.round(np.mean(ys)))
        cx = int(np.round(np.mean(xs)))

        h_half = int(H * zoom_fraction / 2)
        w_half = int(W * zoom_fraction / 2)

        y0 = max(0, cy - h_half)
        y1 = min(H, cy + h_half)
        x0 = max(0, cx - w_half)
        x1 = min(W, cx + w_half)
    else:
        y0, y1 = 0, H
        x0, x1 = 0, W

    fig, ax = plt.subplots(figsize=figsize)

    def update(i):
        ax.clear()

        thr = thresholds[i]
        unc_thr = np.where(unc_z >= thr, unc_z, np.nan)

        ax.imshow(img_z[y0:y1, x0:x1], cmap="gray")

        ax.imshow(
            unc_thr[y0:y1, x0:x1],
            cmap=cmap_unc,
            alpha=alpha,
            vmin=vmin,
            vmax=vmax,
        )

        ax.set_title(
            f"Slice {slice_idx} | Iter {i:02d}\nThreshold = {thr:.6f}"
        )
        ax.set_axis_off()

    ani = FuncAnimation(
        fig,
        update,
        frames=len(thresholds),
        interval=1000 / fps,
        repeat=True,
    )

    savepath = Path(savepath)
    savepath.parent.mkdir(parents=True, exist_ok=True)

    ani.save(savepath, writer=PillowWriter(fps=fps))
    plt.close(fig)

    print(f"Saved GIF to: {savepath}")


make_uncertainty_threshold_sweep_gif(
    img=data.img,
    unc_map=data.unc_map,
    slice_idx=35,
    savepath="uncertainty_threshold_sweep_slice_035.gif",
    fps=2,
    cmap_unc="turbo",
    alpha=0.55,
    zoom_fraction=0.20,
)

Saved GIF to: uncertainty_threshold_sweep_slice_035.gif


In [4]:
unc_handler.threshold_uncertainty_map(unc_threshold=None, target_mm=3.0, method="raycast", mode="median") #unc_threshold=0.033470
unc_handler.compute_band_thickness(method=method)

iter=00 | thr=0.209411 | band=1.41 mm | error=1.59
iter=01 | thr=0.104706 | band=2.46 mm | error=0.54
iter=02 | thr=0.052353 | band=3.28 mm | error=0.28
iter=03 | thr=0.078529 | band=2.93 mm | error=0.07
iter=04 | thr=0.065441 | band=3.05 mm | error=0.05
iter=05 | thr=0.071985 | band=3.05 mm | error=0.05
iter=06 | thr=0.075257 | band=2.99 mm | error=0.01
iter=07 | thr=0.073621 | band=3.05 mm | error=0.05
iter=08 | thr=0.074439 | band=3.05 mm | error=0.05
iter=09 | thr=0.074848 | band=3.05 mm | error=0.05
iter=10 | thr=0.075053 | band=3.05 mm | error=0.05
iter=11 | thr=0.075155 | band=3.05 mm | error=0.05
iter=12 | thr=0.075206 | band=2.99 mm | error=0.01
iter=13 | thr=0.075181 | band=3.05 mm | error=0.05
iter=14 | thr=0.075193 | band=3.05 mm | error=0.05
iter=15 | thr=0.075200 | band=3.05 mm | error=0.05
iter=16 | thr=0.075203 | band=2.99 mm | error=0.01
iter=17 | thr=0.075201 | band=3.05 mm | error=0.05
iter=18 | thr=0.075202 | band=3.05 mm | error=0.05
iter=19 | thr=0.075203 | band=3

In [5]:
def plot_band_density_diagnostics(unc_handler, target_mm=3.0, xlim=(0, 25)):
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    all_values = np.array(unc_handler.all_records, dtype=float)
    slice_value_records = unc_handler.slice_value_records

    if len(all_values) < 2:
        raise ValueError("Not enough values found. Run threshold_uncertainty_map(...) first.")

    slice_means = np.array([np.mean(v) for v in slice_value_records if len(v) > 0], dtype=float)
    slice_medians = np.array([np.median(v) for v in slice_value_records if len(v) > 0], dtype=float)

    slice_options = ["All"] + list(range(len(slice_value_records)))

    slice_dropdown = widgets.Dropdown(
        options=slice_options,
        value="All",
        description="Slice:"
    )

    out = widgets.Output()

    def add_reference_lines(mean_val, median_val):
        plt.axvline(
            mean_val,
            color="tab:blue",
            linestyle="--",
            linewidth=3,
            alpha=0.65,
            label=f"Mean = {mean_val:.2f}"
        )

        plt.axvline(
            median_val,
            color="tab:orange",
            linestyle=":",
            linewidth=3,
            alpha=0.65,
            label=f"Median = {median_val:.2f}"
        )

        plt.axvline(
            target_mm,
            color="tab:red",
            linestyle="-",
            linewidth=3,
            alpha=0.55,
            label=f"Target = {target_mm:.2f}"
        )

    def _plot(selected_slice):
        with out:
            clear_output(wait=True)

            if selected_slice == "All":
                values = all_values
                title = "Density of all pooled band thickness values"
            else:
                values = np.array(slice_value_records[selected_slice], dtype=float)
                title = f"Density of band thickness values, stored slice {selected_slice}"

            if len(values) < 2:
                print("Not enough values to create a density plot.")
                return

            mean_val = np.mean(values)
            median_val = np.median(values)

            plt.figure(figsize=(8, 5))
            pd.Series(values).plot(kind="density", label="Density")

            add_reference_lines(mean_val, median_val)

            plt.xlim(*xlim)
            plt.ylim(0,0.30)
            plt.xlabel("Band thickness [mm]")
            plt.ylabel("Density")
            plt.title(title)
            plt.legend()
            plt.grid(True)

            print("Selected value distribution")
            print(f"Mean   : {mean_val:.3f} mm")
            print(f"Median : {median_val:.3f} mm")
            print(f"Std    : {np.std(values):.3f} mm")
            print(f"N      : {len(values)}")

            plt.show()

    widgets.interactive_output(_plot, {"selected_slice": slice_dropdown})
    display(slice_dropdown, out)

In [ ]:
plot_band_density_diagnostics(unc_handler)

Dropdown(description='Slice:', options=('All', 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 1…

Output()

In [7]:
nietjes_prompts = unc_handler.generate_prompts_nietjes(unc_band_thr_mm=4.0,
        interpix_dist=4,
        pixel_interval=20,
        angle_step=5,
        method=method)

print(nietjes_prompts)

bbox_prompts = unc_handler.generate_prompts_boxes(band_threshold=0.0)

print(bbox_prompts)

23 n_results: 8 max_total: 14.650000259280205 max_inner: 11.48560020327568 max_outer: 3.1644000560045242
24 n_results: 8 max_total: 9.844800174236298 max_inner: 1.9924000352621078 max_outer: 9.493200168013573
25 n_results: 8 max_total: 12.657600224018097 max_inner: 1.7580000311136246 max_outer: 11.720000207424164
26 n_results: 8 max_total: 9.141600161790848 max_inner: 3.633200064301491 max_outer: 7.383600130677223
27 n_results: 8 max_total: 9.610400170087814 max_inner: 3.9848000705242157 max_outer: 8.321200147271156
28 n_results: 8 max_total: 8.55560015141964 max_inner: 2.226800039410591 max_outer: 7.266400128602982
29 n_results: 8 max_total: 6.3288001120090485 max_inner: 2.6956000477075577 max_outer: 4.6880000829696655
30 n_results: 8 max_total: 3.633200064301491 max_inner: 2.226800039410591 max_outer: 3.0472000539302826
31 n_results: 8 max_total: 2.4612000435590744 max_inner: 1.289200022816658 max_outer: 1.9924000352621078
32 n_results: 8 max_total: 1.9924000352621078 max_inner: 0.93

In [8]:
# PLOTTING FUNCTION FOR IMAGE, SEGMENTATION, UNCERTAINTY, PROMPTS, NORMALS/RAYS

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from ipywidgets import interact, IntSlider, Checkbox
from matplotlib.colors import ListedColormap


def plot_ug_prompts_interactive(
    ug_instance,
    alpha_seg=0.35,
    alpha_unc=0.35,
    figsize=(7, 7),
    show_only_prompted_slices=False,
    unc_mode="binary",
    zoom_fraction=0.25,
    prompt_dict = None
):
    """
    Interactive slice viewer with optional zoom.

    Toggleable overlays:
    - uncertainty
    - vectors (normals OR rays)
    """

    if not hasattr(ug_instance, "img"):
        raise AttributeError("The instance has no attribute 'img'.")
    if not hasattr(ug_instance, "mask"):
        raise AttributeError("The instance has no attribute 'mask'.")
    if not hasattr(ug_instance, "prompts_by_slice"):
        raise AttributeError(
            "The instance has no attribute 'prompts_by_slice'. "
            "Run a prompt generation function first."
        )

    img = ug_instance.img
    mask = ug_instance.mask
    prompts_by_slice = ug_instance.prompts_by_slice

    if hasattr(ug_instance, "unc_map_bin"):
        unc = ug_instance.unc_map_bin.astype(bool)
    elif hasattr(ug_instance, "unc_map"):
        unc = ug_instance.unc_map
    else:
        unc = None

    n_slices = img.shape[0]

    if show_only_prompted_slices:
        slice_list = sorted(prompts_by_slice.keys())
        if len(slice_list) == 0:
            print("No prompted slices found.")
            return
    else:
        slice_list = list(range(n_slices))

    def _plot(slice_idx_in_list, show_uncertainty, show_vectors, zoom):
        z = slice_list[slice_idx_in_list]

        H, W = img[z].shape

        # --- define crop ---
        if zoom:
            h_half = int(H * zoom_fraction / 2)
            w_half = int(W * zoom_fraction / 2)
            cy, cx = H // 2, W // 2
            y0, y1 = cy - h_half, cy + h_half
            x0, x1 = cx - w_half, cx + w_half
        else:
            y0, y1 = 0, H
            x0, x1 = 0, W

        def shift_points(points):
            return np.column_stack([
                points[:, 0] - x0,
                points[:, 1] - y0
            ])

        fig, ax = plt.subplots(figsize=figsize)

        # --- image ---
        img_crop = img[z][y0:y1, x0:x1]
        ax.imshow(img_crop, cmap="gray")

        # --- segmentation ---
        seg_crop = mask[z][y0:y1, x0:x1]
        seg_overlay = np.ma.masked_where(~seg_crop, seg_crop)
        ax.imshow(seg_overlay,cmap=ListedColormap(["red"]), alpha=alpha_seg)

        # --- uncertainty ---
        if show_uncertainty and unc is not None:
            unc_slice = unc[z][y0:y1, x0:x1]

            if unc_mode == "binary":
                unc_overlay = np.ma.masked_where(~unc_slice, unc_slice)
            else:
                unc_overlay = np.ma.masked_where(unc_slice <= 0, unc_slice)

            ax.imshow(unc_overlay, cmap = ListedColormap(["deepskyblue"]), alpha=alpha_unc)

        # --- prompts ---
        prompts = prompt_dict.get(z, {})

        points = prompts.get("points", None)
        labels = prompts.get("point_labels", None)

        if points is not None and labels is not None:
            points = np.asarray(points)
            labels = np.asarray(labels)

            pos = points[labels == 1]
            neg = points[labels == 0]

            if len(pos) > 0:
                ax.scatter(
                    pos[:, 0] - x0,
                    pos[:, 1] - y0,
                    marker="+",
                    s=25,
                    linewidths=1.5,
                    color="lime",
                )

            if len(neg) > 0:
                ax.scatter(
                    neg[:, 0] - x0,
                    neg[:, 1] - y0,
                    marker="x",
                    s=25,
                    linewidths=1.3,
                    color="red",
                )

        # --- boxes ---
        # boxes = prompts.get("boxes", None)
        # if boxes is None:
        #     boxes = prompts.get("bbox", None)

        # if boxes is not None:
        #     boxes = np.asarray(boxes)
        #     if boxes.ndim == 1:
        #         boxes = boxes[None, :]

        #     for i, (bx0, by0, bx1, by1) in enumerate(boxes):
        #         rect = Rectangle(
        #             (bx0 - x0, by0 - y0),
        #             bx1 - bx0,
        #             by1 - by0,
        #             fill=False,
        #             edgecolor="cyan",
        #             linewidth=1,
        #             label="box" if i == 0 else None,
        #         )
        #         ax.add_patch(rect)

        # --- VECTORS: normals OR rays ---
        if show_vectors:

            # ---------- NORMALS ----------
            normals_data = getattr(ug_instance, "normals_by_slice", {}).get(z, None)

            if normals_data is not None:
                mids = np.asarray(normals_data["midpoints"])  # (y, x)

                if "outer_normals" in normals_data:
                    norms = np.asarray(normals_data["outer_normals"])
                elif "normals" in normals_data:
                    norms = np.asarray(normals_data["normals"])
                else:
                    norms = None

                if norms is not None:
                    mask_crop = (
                        (mids[:, 0] >= y0) & (mids[:, 0] < y1) &
                        (mids[:, 1] >= x0) & (mids[:, 1] < x1)
                    )

                    mids = mids[mask_crop]
                    norms = norms[mask_crop]

                    if len(mids) > 0:
                        mids_plot = np.column_stack([
                            mids[:, 1] - x0,
                            mids[:, 0] - y0,
                        ])

                        arrow_length = 15

                        norms_plot = norms.copy().astype(float)
                        norms_plot /= (
                            np.linalg.norm(norms_plot, axis=1, keepdims=True) + 1e-8
                        )
                        norms_plot *= arrow_length

                        ax.quiver(
                            mids_plot[:, 0],
                            mids_plot[:, 1],
                            norms_plot[:, 1],
                            norms_plot[:, 0],
                            color="cyan",
                            scale=1,
                            width=0.003,
                            alpha=0.7,
                            angles="xy",
                            scale_units="xy",
                            label="normals",
                        )

            # ---------- RAYS ----------
            rays_data = getattr(ug_instance, "rays_by_slice", {}).get(z, None)

            if rays_data is not None:
                origin_yx = np.asarray(rays_data["origin_yx"], dtype=float)
                dirs_yx = np.asarray(rays_data["directions_yx"], dtype=float)

                seg_mm = np.asarray(
                    rays_data.get("seg_mm", np.ones(len(dirs_yx)) * 20),
                    dtype=float
                )

                spacing_y = ug_instance.img_spacing[1]
                spacing_x = ug_instance.img_spacing[2]

                # --- normalize directions ---
                dirs_norm = dirs_yx / (np.linalg.norm(dirs_yx, axis=1, keepdims=True) + 1e-8)

                # --- convert origin to plot coords ---
                origin_plot = np.array([
                    origin_yx[1] - x0,
                    origin_yx[0] - y0,
                ])

                ax.scatter(
                    origin_plot[0],
                    origin_plot[1],
                    s=20,
                    c="cyan",
                    marker="o",
                    label="ray origin",
                )

                # --- control how far arrows extend ---
                extend_mm = 20.0  # extra length beyond segmentation

                # --- convert lengths to pixel space ---
                lengths_px = np.column_stack([
                    (seg_mm + extend_mm) / spacing_y,  # y
                    (seg_mm + extend_mm) / spacing_x,  # x
                ])

                # --- compute arrow vectors in pixel space ---
                vecs_y = dirs_norm[:, 0] * lengths_px[:, 0]
                vecs_x = dirs_norm[:, 1] * lengths_px[:, 1]

                # --- plot arrows ---
                ax.quiver(
                    np.full_like(vecs_x, origin_plot[0]),  # x start
                    np.full_like(vecs_y, origin_plot[1]),  # y start
                    vecs_x,                                # dx
                    vecs_y,                                # dy
                    color="cyan",
                    angles="xy",
                    scale_units="xy",
                    scale=1,
                    width=0.003,
                    alpha=0.6,
                    label="rays"
                )

        # --- title ---
        has_prompt = z in prompts_by_slice
        title = f"Slice {z} | prompted: {has_prompt}"

        if (
            hasattr(ug_instance, "band_thickness_per_slice")
            and z < len(ug_instance.band_thickness_per_slice)
        ):
            band = ug_instance.band_thickness_per_slice[z]
            title += f" | band: {band:.2f} mm"

        ax.set_title(title)
        ax.set_axis_off()

        handles, labels = ax.get_legend_handles_labels()
        if handles:
            unique = dict(zip(labels, handles))
            ax.legend(unique.values(), unique.keys(), loc="upper right")

        plt.tight_layout()
        plt.show()

    interact(
        _plot,
        slice_idx_in_list=IntSlider(
            min=0,
            max=len(slice_list) - 1,
            step=1,
            value=0,
            description="slice",
        ),
        show_uncertainty=Checkbox(value=False, description="uncertainty"),
        show_vectors=Checkbox(value=False, description="vectors"),
        zoom=Checkbox(value=False, description="zoom"),
    )

In [9]:
plot_ug_prompts_interactive(unc_handler,unc_mode="abs",prompt_dict=combine_prompt_sets([nietjes_prompts,bbox_prompts]))

interactive(children=(IntSlider(value=0, description='slice', max=87), Checkbox(value=False, description='unce…

In [10]:
propagation_styles = ['default', 'full', 'prompt_based', 'central_start', 'central_partitions']
propagation_style = propagation_styles[3]

dense_prompt = seg_handler.load_dense_prompt()
dense_and_nietjes_prompts = combine_prompt_sets(prompt_dict_list = [dense_prompt, nietjes_prompts])
all = combine_prompt_sets(prompt_dict_list = [nietjes_prompts, bbox_prompts])


prompt_sets = [dense_and_nietjes_prompts, bbox_prompts, all]
prompt_names = ["Dense_and_nietjes", "Uncertainty_bboxes","All"]
prompt_weights = [0.5 , 0.4, 0.1]

seg_handler.compile_prompt_sets(prompt_dict_list=prompt_sets, prompt_set_names=prompt_names, prompt_set_weights=prompt_weights)

seg_handler.run_segmentation_sets(propagation_style=propagation_style, weighting_strategy="custom", threshold=0.0, no_external_propagation=False)
seg_handler.remove_distant_slices(tolerance_frames=0)

Running segmentation for prompt set 'Dense_and_nietjes' with slices: [23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 47, 48, 49, 50]
Device: cuda
Volume shape (D,H,W): (88, 1024, 1024)
Using prompts from slices: [23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 47, 48, 49, 50]
Adding prompt(s) on slice 23


c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\sam2_video_predictor_npz.py:965: UserWarning: cannot import name '_C' from 'sam2' (c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(


Adding prompt(s) on slice 24
Adding prompt(s) on slice 25
Adding prompt(s) on slice 26
Adding prompt(s) on slice 27
Adding prompt(s) on slice 28
Adding prompt(s) on slice 29
Adding prompt(s) on slice 30
Adding prompt(s) on slice 31
Adding prompt(s) on slice 32
Adding prompt(s) on slice 33
Adding prompt(s) on slice 34
Adding prompt(s) on slice 35
Adding prompt(s) on slice 36
Adding prompt(s) on slice 37
Adding prompt(s) on slice 38
Adding prompt(s) on slice 39
Adding prompt(s) on slice 40
Adding prompt(s) on slice 41
Adding prompt(s) on slice 42
Adding prompt(s) on slice 43
Adding prompt(s) on slice 44
Adding prompt(s) on slice 45
Adding prompt(s) on slice 47
Adding prompt(s) on slice 48
Adding prompt(s) on slice 49
Adding prompt(s) on slice 50
Forward propagation (central_start, from slice 36)...


propagate in video: 100%|██████████| 52/52 [00:14<00:00,  3.54it/s]


Backward propagation (central_start, from slice 36)...


...

propagate in video: 100%|██████████| 52/52 [00:15<00:00,  3.46it/s]


Backward propagation (central_start, from slice 36)...


propagate in video: 100%|██████████| 37/37 [00:09<00:00,  3.95it/s]


Kept slices 23 to 50. Removed predicted segmentation outside dense mask ±0 slices.


array([[[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       ...,

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 

In [11]:
# PLOTTING FUNCTION FOR GT VS DENSE MASK VS UPDATED SEGMENTATION + PROMPTS

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Checkbox
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
from skimage.measure import find_contours


def plot_segmentation_overlays_interactive(
    seg_instance,
    figsize=(7, 7),
    alpha_gt=0.45,
    alpha_dense_mask=0.35,
    alpha_seg=0.45,
    zoom_fraction=0.25,
):
    """
    Interactive viewer for image, ground truth, dense prompt mask,
    newly generated segmentation, point prompts, and bbox prompts.

    Added:
    - Toggle between filled masks and contour-only display.
    """

    if not hasattr(seg_instance, "img"):
        raise AttributeError("seg_instance has no attribute 'img'.")

    if not hasattr(seg_instance, "gt"):
        raise AttributeError("seg_instance has no attribute 'gt'.")

    if not hasattr(seg_instance, "predicted_seg"):
        raise AttributeError(
            "seg_instance has no attribute 'predicted_seg'. "
            "Run seg_instance.run_segmentation() first."
        )

    img = seg_instance.img
    gt = seg_instance.gt.astype(bool)
    pred = seg_instance.predicted_seg.astype(bool)

    prompts_by_slice = getattr(seg_instance, "dense_prompt_set", {})

    if img.shape != gt.shape or img.shape != pred.shape:
        raise ValueError(
            f"Shape mismatch: img={img.shape}, gt={gt.shape}, pred={pred.shape}"
        )

    n_slices = img.shape[0]

    color_gt = "#57CC99"
    color_dense = "#4A90E2"
    color_pred = "#F4A261"

    gt_cmap = ListedColormap([(0, 0, 0, 0), color_gt])
    dense_cmap = ListedColormap([(0, 0, 0, 0), color_dense])
    pred_cmap = ListedColormap([(0, 0, 0, 0), color_pred])

    def unpack_prompt(prompt):
        if isinstance(prompt, dict):
            return (
                prompt.get("points", None),
                prompt.get("point_labels", None),
                prompt.get("bbox", None),
                prompt.get("mask_input", None),
            )

        return prompt

    def shift_points(points, x0, y0):
        return np.column_stack([
            points[:, 0] - x0,
            points[:, 1] - y0,
        ])

    def draw_contour(ax, mask, color, linewidth=1.8, label=None):
        contours = find_contours(mask.astype(float), level=0.5)

        for i, contour in enumerate(contours):
            ax.plot(
                contour[:, 1],
                contour[:, 0],
                color=color,
                linewidth=linewidth,
                label=label if i == 0 else None,
            )

    def _plot(
        slice_idx,
        zoom,
        show_as_contours,
        show_ground_truth,
        show_dense_mask,
        show_updated_segmentation,
        show_point_prompts,
        show_bbox_prompts,
    ):
        z = slice_idx
        H, W = img[z].shape

        prompt_data = prompts_by_slice.get(z, None)

        if zoom:
            h_half = int(H * zoom_fraction / 2)
            w_half = int(W * zoom_fraction / 2)

            fg = gt[z] | pred[z]

            if prompt_data is not None:
                points, point_labels, bbox, mask_input = unpack_prompt(prompt_data)

                if mask_input is not None:
                    fg = fg | mask_input.astype(bool)

                if points is not None and len(points) > 0:
                    points_arr = np.asarray(points)
                    px = points_arr[:, 0]
                    py = points_arr[:, 1]

                    prompt_mask = np.zeros_like(fg, dtype=bool)

                    valid = (
                        (py >= 0) & (py < H) &
                        (px >= 0) & (px < W)
                    )

                    prompt_mask[
                        py[valid].astype(int),
                        px[valid].astype(int),
                    ] = True

                    fg = fg | prompt_mask

            if fg.any():
                ys, xs = np.where(fg)
                cy = int(np.round(np.mean(ys)))
                cx = int(np.round(np.mean(xs)))
            else:
                cy, cx = H // 2, W // 2

            y0 = max(0, cy - h_half)
            y1 = min(H, cy + h_half)
            x0 = max(0, cx - w_half)
            x1 = min(W, cx + w_half)

        else:
            y0, y1 = 0, H
            x0, x1 = 0, W

        img_crop = img[z][y0:y1, x0:x1]
        gt_crop = gt[z][y0:y1, x0:x1]
        pred_crop = pred[z][y0:y1, x0:x1]

        fig, ax = plt.subplots(figsize=figsize)

        ax.imshow(img_crop, cmap="gray")

        # ===================== GT =====================

        if show_ground_truth:

            if show_as_contours:
                draw_contour(
                    ax,
                    gt_crop,
                    color_gt,
                    linewidth=2.0,
                    label="Ground truth",
                )

            else:
                gt_overlay = np.ma.masked_where(
                    ~gt_crop,
                    gt_crop.astype(np.uint8),
                )

                ax.imshow(
                    gt_overlay,
                    cmap=gt_cmap,
                    alpha=alpha_gt,
                    vmin=0,
                    vmax=1,
                )

        # ===================== DENSE MASK =====================

        if show_dense_mask:

            if prompt_data is not None:
                _, _, _, mask_input = unpack_prompt(prompt_data)

                if mask_input is not None:

                    dense_crop = mask_input[y0:y1, x0:x1].astype(bool)

                    if show_as_contours:

                        draw_contour(
                            ax,
                            dense_crop,
                            color_dense,
                            linewidth=2.0,
                            label="Dense mask prompt",
                        )

                    else:

                        dense_overlay = np.ma.masked_where(
                            ~dense_crop,
                            dense_crop.astype(np.uint8),
                        )

                        ax.imshow(
                            dense_overlay,
                            cmap=dense_cmap,
                            alpha=alpha_dense_mask,
                            vmin=0,
                            vmax=1,
                        )

        # ===================== PRED =====================

        if show_updated_segmentation:

            if show_as_contours:

                draw_contour(
                    ax,
                    pred_crop,
                    color_pred,
                    linewidth=2.0,
                    label="Updated segmentation",
                )

            else:

                pred_overlay = np.ma.masked_where(
                    ~pred_crop,
                    pred_crop.astype(np.uint8),
                )

                ax.imshow(
                    pred_overlay,
                    cmap=pred_cmap,
                    alpha=alpha_seg,
                    vmin=0,
                    vmax=1,
                )

        # ===================== POINTS =====================

        if prompt_data is not None:

            points, point_labels, bbox, mask_input = unpack_prompt(prompt_data)

            if show_point_prompts and points is not None and point_labels is not None:

                points = np.asarray(points)
                point_labels = np.asarray(point_labels)

                if len(points) > 0:

                    pos = points[point_labels == 1]
                    neg = points[point_labels == 0]

                    if len(pos) > 0:

                        pos = shift_points(pos, x0, y0)

                        ax.scatter(
                            pos[:, 0],
                            pos[:, 1],
                            s=20,
                            c="lightgreen",
                            marker="+",
                            linewidths=1.5,
                            label="positive prompt",
                        )

                    if len(neg) > 0:

                        neg = shift_points(neg, x0, y0)

                        ax.scatter(
                            neg[:, 0],
                            neg[:, 1],
                            s=20,
                            c="lightcoral",
                            marker="x",
                            linewidths=1.5,
                            label="negative prompt",
                        )

            # ===================== BBOX =====================

            if show_bbox_prompts and bbox is not None:

                boxes = np.asarray(bbox)

                if boxes.ndim == 1:
                    boxes = boxes[None, :]

                for i, (bx0, by0, bx1, by1) in enumerate(boxes):

                    rect = Rectangle(
                        (bx0 - x0, by0 - y0),
                        bx1 - bx0,
                        by1 - by0,
                        fill=False,
                        edgecolor="cyan",
                        linewidth=1.2,
                        label="bbox prompt" if i == 0 else None,
                    )

                    ax.add_patch(rect)

        title = f"Slice {z}"

        if show_ground_truth:
            title += f" | GT: {int(gt_crop.sum())}"

        if show_dense_mask:
            title += " | Dense mask"

        if show_updated_segmentation:
            title += f" | Seg: {int(pred_crop.sum())}"

        if show_as_contours:
            title += " | Contours"

        ax.set_title(title)

        ax.set_axis_off()

        handles = []

        if show_ground_truth:
            handles.append(
                mpatches.Patch(color=color_gt, label="Ground truth")
            )

        if show_dense_mask:
            handles.append(
                mpatches.Patch(color=color_dense, label="Dense mask prompt")
            )

        if show_updated_segmentation:
            handles.append(
                mpatches.Patch(color=color_pred, label="Updated segmentation")
            )

        existing_handles, existing_labels = ax.get_legend_handles_labels()

        handles.extend(existing_handles)

        if len(handles) > 0:

            labels = [h.get_label() for h in handles]

            unique = dict(zip(labels, handles))

            ax.legend(
                unique.values(),
                unique.keys(),
                loc="upper right",
                framealpha=0.85,
            )

        plt.tight_layout()
        plt.show()

    interact(
        _plot,
        slice_idx=IntSlider(
            min=0,
            max=n_slices - 1,
            step=1,
            value=0,
            description="slice",
        ),
        zoom=Checkbox(value=False, description="zoom"),
        show_as_contours=Checkbox(
            value=False,
            description="contours only",
        ),
        show_ground_truth=Checkbox(
            value=True,
            description="ground truth",
        ),
        show_dense_mask=Checkbox(
            value=True,
            description="dense mask",
        ),
        show_updated_segmentation=Checkbox(
            value=True,
            description="segmentation",
        ),
        show_point_prompts=Checkbox(
            value=True,
            description="points",
        ),
        show_bbox_prompts=Checkbox(
            value=True,
            description="bbox",
        ),
    )

In [12]:
plot_segmentation_overlays_interactive(seg_handler)

interactive(children=(IntSlider(value=0, description='slice', max=87), Checkbox(value=False, description='zoom…

In [13]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Checkbox
import matplotlib.patches as mpatches
from skimage.measure import find_contours


def plot_prompt_sets_contours_interactive(
    seg_instance,
    figsize=(7, 7),
    zoom_fraction=0.25,
):
    """
    Interactive contour viewer for all prompt-set segmentations.

    Shows:
    - image
    - ground truth contour
    - final fused segmentation contour
    - one toggle per prompt set from seg_instance.segs_per_set
    """

    if not hasattr(seg_instance, "img"):
        raise AttributeError("seg_instance has no attribute 'img'.")

    if not hasattr(seg_instance, "gt"):
        raise AttributeError("seg_instance has no attribute 'gt'.")

    if not hasattr(seg_instance, "segs_per_set"):
        raise AttributeError(
            "seg_instance has no attribute 'segs_per_set'. "
            "Run run_segmentation_sets() first."
        )

    img = seg_instance.img
    gt = seg_instance.gt.astype(bool)

    segs_per_set = {
        name: seg.astype(bool)
        for name, seg in seg_instance.segs_per_set.items()
    }

    final_seg = None
    if hasattr(seg_instance, "predicted_seg"):
        final_seg = seg_instance.predicted_seg.astype(bool)

    if img.shape != gt.shape:
        raise ValueError(f"Shape mismatch: img={img.shape}, gt={gt.shape}")

    for name, seg in segs_per_set.items():
        if seg.shape != img.shape:
            raise ValueError(
                f"Shape mismatch for {name}: img={img.shape}, seg={seg.shape}"
            )

    if final_seg is not None and final_seg.shape != img.shape:
        raise ValueError(
            f"Shape mismatch for final segmentation: img={img.shape}, final_seg={final_seg.shape}"
        )

    n_slices = img.shape[0]

    strategy_names = list(segs_per_set.keys())

    colors = [
        "#F4A261",
        "#2A9D8F",
        "#E76F51",
        "#9B5DE5",
        "#00BBF9",
        "#F15BB5",
        "#90BE6D",
        "#577590",
    ]

    color_gt = "#57CC99"
    color_final = "#FFD166"

    strategy_color_dict = {}

    for i, name in enumerate(strategy_names):
        strategy_color_dict[name] = colors[i % len(colors)]

    def draw_contour(ax, mask, color, linewidth=2.0, label=None):
        contours = find_contours(mask.astype(float), level=0.5)

        for i, contour in enumerate(contours):
            ax.plot(
                contour[:, 1],
                contour[:, 0],
                color=color,
                linewidth=linewidth,
                label=label if i == 0 else None,
            )

    def _plot(
        slice_idx,
        zoom,
        show_ground_truth,
        show_final_segmentation,
        **prompt_set_toggles,
    ):
        z = slice_idx
        H, W = img[z].shape

        if zoom:
            h_half = int(H * zoom_fraction / 2)
            w_half = int(W * zoom_fraction / 2)

            fg = gt[z].copy()

            for name in strategy_names:
                if prompt_set_toggles.get(name, False):
                    fg = fg | segs_per_set[name][z]

            if show_final_segmentation and final_seg is not None:
                fg = fg | final_seg[z]

            if fg.any():
                ys, xs = np.where(fg)
                cy = int(np.round(np.mean(ys)))
                cx = int(np.round(np.mean(xs)))
            else:
                cy, cx = H // 2, W // 2

            y0 = max(0, cy - h_half)
            y1 = min(H, cy + h_half)
            x0 = max(0, cx - w_half)
            x1 = min(W, cx + w_half)

        else:
            y0, y1 = 0, H
            x0, x1 = 0, W

        img_crop = img[z][y0:y1, x0:x1]
        gt_crop = gt[z][y0:y1, x0:x1]

        fig, ax = plt.subplots(figsize=figsize)
        ax.imshow(img_crop, cmap="gray")

        handles = []

        if show_ground_truth:
            draw_contour(
                ax,
                gt_crop,
                color=color_gt,
                linewidth=2.4,
                label="Ground truth",
            )
            handles.append(mpatches.Patch(color=color_gt, label="Ground truth"))

        for name in strategy_names:
            if prompt_set_toggles.get(name, False):

                seg_crop = segs_per_set[name][z][y0:y1, x0:x1]
                color = strategy_color_dict[name]

                draw_contour(
                    ax,
                    seg_crop,
                    color=color,
                    linewidth=2.0,
                    label=name,
                )

                handles.append(mpatches.Patch(color=color, label=name))

        if show_final_segmentation and final_seg is not None:
            final_crop = final_seg[z][y0:y1, x0:x1]

            draw_contour(
                ax,
                final_crop,
                color=color_final,
                linewidth=2.6,
                label="Final fused segmentation",
            )

            handles.append(
                mpatches.Patch(color=color_final, label="Final fused segmentation")
            )

        title = f"Slice {z}"

        if show_ground_truth:
            title += f" | GT: {int(gt_crop.sum())}"

        for name in strategy_names:
            if prompt_set_toggles.get(name, False):
                seg_crop = segs_per_set[name][z][y0:y1, x0:x1]
                title += f" | {name}: {int(seg_crop.sum())}"

        if show_final_segmentation and final_seg is not None:
            final_crop = final_seg[z][y0:y1, x0:x1]
            title += f" | Final: {int(final_crop.sum())}"

        ax.set_title(title)
        ax.set_axis_off()

        existing_handles, existing_labels = ax.get_legend_handles_labels()
        handles.extend(existing_handles)

        if len(handles) > 0:
            labels = [h.get_label() for h in handles]
            unique = dict(zip(labels, handles))

            ax.legend(
                unique.values(),
                unique.keys(),
                loc="upper right",
                framealpha=0.85,
            )

        plt.tight_layout()
        plt.show()

    controls = {
        "slice_idx": IntSlider(
            min=0,
            max=n_slices - 1,
            step=1,
            value=0,
            description="slice",
        ),
        "zoom": Checkbox(value=False, description="zoom"),
        "show_ground_truth": Checkbox(value=True, description="ground truth"),
        "show_final_segmentation": Checkbox(
            value=True,
            description="final fused",
        ),
    }

    for name in strategy_names:
        controls[name] = Checkbox(
            value=True,
            description=name,
        )

    interact(_plot, **controls)

In [14]:
plot_prompt_sets_contours_interactive(seg_handler)

interactive(children=(IntSlider(value=0, description='slice', max=87), Checkbox(value=False, description='zoom…

In [15]:
if run_eval:

    eval_handler = Evaluator(segmentation=seg_handler)
    metrics = eval_handler.compute_all(surface_dice_tol=1.0)

    nn_unet_eval = Evaluator(
        pred=seg_handler.mask,
        gt=seg_handler.gt,
        spacing=seg_handler.img_spacing,
    )
    nn_unet_metrics = nn_unet_eval.compute_all(surface_dice_tol=1.0)


    print("\n" + "=" * 105)
    print(f"{'Metric':<40} {'nnUNet Dense Mask':>22} {'Updated Segmentation':>28}")
    print("=" * 105)

    for key in nn_unet_metrics.keys():

        nn_val = nn_unet_metrics[key]
        new_val = metrics[key]

        # Handle strings separately (e.g. subject name)
        if isinstance(nn_val, str) or isinstance(new_val, str):

            print(
                f"{key:<40} "
                f"{str(nn_val):>22} "
                f"{str(new_val):>28}"
            )

        else:

            print(
                f"{key:<40} "
                f"{nn_val:>22.4f} "
                f"{new_val:>28.4f}"
            )

    print("=" * 105)


Metric                                        nnUNet Dense Mask         Updated Segmentation
subject name                                                  x      newAcq_0b4940fa31a1d650
HD_mm                                                   13.8231                      12.4425
HD95_mm                                                  5.9102                       5.6984
MSD_mm                                                   2.0008                       2.2098
ASSD_mm                                                  1.9475                       1.9447
Dice                                                     0.8410                       0.8590
SurfaceDice@1.0mm                                        0.3954                       0.3842
CentroidDistance_mm                                      5.3147                       4.5272
PredictionVolume_mm3                                 68961.0619                   72619.1909
GroundTruthVolume_mm3                                71034.0749      

In [16]:
if evaluate_compare_recontours:
    data.load_recontours()
    
    results_df, consensus, vote_map = compare_to_recontours(
    pred_seg=seg_handler.predicted_seg,
    recontours=data.observer_recontours,
    observer_names=data.observer_names,
    spacing=data.img_spacing,
    subject_name=data.subjectfolder.name,
    )

#PRINTING FUNCTION FOR GENERTED METRICS COMPARED TO RECONTOURS
def print_recontour_summary(results_df):

    obs = results_df[results_df["Group"] == "Clinician vs clinician"]
    pred = results_df[results_df["Group"] == "Prediction vs clinician"]

    print("\n=== Inter-observer variability ===")
    print(
        f"Surface Dice : {obs.iloc[:,2].mean():.3f} ± {obs.iloc[:,2].std():.3f}"
    )
    print(
        f"ASSD         : {obs['ASSD_mm'].mean():.3f} ± {obs['ASSD_mm'].std():.3f} mm"
    )
    print(
        f"HD95         : {obs['HD95_mm'].mean():.3f} ± {obs['HD95_mm'].std():.3f} mm"
    )

    print("\n=== Prediction vs clinicians ===")
    print(
        f"Surface Dice : {pred.iloc[:,2].mean():.3f} ± {pred.iloc[:,2].std():.3f}"
    )
    print(
        f"ASSD         : {pred['ASSD_mm'].mean():.3f} ± {pred['ASSD_mm'].std():.3f} mm"
    )
    print(
        f"HD95         : {pred['HD95_mm'].mean():.3f} ± {pred['HD95_mm'].std():.3f} mm"
    )

In [17]:
display(results_df.round(3))

print_recontour_summary(results_df)

,Group,Comparison,SurfaceDice@1.0mm,ASSD_mm,HD95_mm,CentroidDistance_mm,RelativeVolumeDifference_percent
0,Clinician vs clinician,B vs C,0.909,0.262,1.406,0.154,-3.400
1,Clinician vs clinician,B vs D,0.880,0.326,1.482,0.601,-4.400
2,Clinician vs clinician,B vs E,0.887,0.310,1.690,0.328,-3.446
3,Clinician vs clinician,C vs D,0.915,0.224,1.406,0.510,-1.035
4,Clinician vs clinician,C vs E,0.894,0.258,1.690,0.375,-0.047
5,Clinician vs clinician,D vs E,0.866,0.337,2.344,0.870,0.999
6,Prediction vs clinician,Prediction vs B,0.668,1.048,4.125,2.718,-2.707
7,Prediction vs clinician,Prediction vs C,0.641,1.101,4.125,2.734,-6.016
8,Prediction vs clinician,Prediction vs D,0.614,1.247,4.894,3.183,-6.989
9,Prediction vs clinician,Prediction vs E,0.689,0.929,3.570,2.391,-6.060



=== Inter-observer variability ===
Surface Dice : 0.892 ± 0.018
ASSD         : 0.286 ± 0.045 mm
HD95         : 1.670 ± 0.355 mm

=== Prediction vs clinicians ===
Surface Dice : 0.653 ± 0.032
ASSD         : 1.081 ± 0.132 mm
HD95         : 4.179 ± 0.544 mm


In [18]:
#PLOTTING FUNCTION FOR GT VS ORIGINAL SEGMENTATION VS UPDATED SEGMENTATION + RECONTOURS

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Checkbox
import matplotlib.patches as mpatches
from skimage.measure import find_contours


def plot_recontours_interactive(
    img,
    gt,
    original_seg,
    pred_seg,
    recontours,
    observer_names=None,
    figsize=(7, 7),
    zoom_fraction=0.30,
):
    """
    Interactive contour viewer for:
    - Ground truth (soft green)
    - Original segmentation (soft blue)
    - Generated segmentation (soft orange)
    - Observer recontours (red shades)
    """

    img = np.asarray(img)
    gt = np.asarray(gt).astype(bool)
    original_seg = np.asarray(original_seg).astype(bool)
    pred_seg = np.asarray(pred_seg).astype(bool)
    recontours = [np.asarray(r).astype(bool) for r in recontours]

    if observer_names is None:
        observer_names = [f"{i+1}" for i in range(len(recontours))]

    n_slices = img.shape[0]

    # Softer colors
    color_gt = "#7fc97f"
    color_original = "#80b1d3"
    color_pred = "#fdb462"

    # Red shades for observers
    observer_colors = [
        "#ffcccc",
        "#ff8a8a",
        "#e63946",
        "#9d0208",
    ]

    def draw_contour(ax, mask, color, linewidth=1.8):
        contours = find_contours(mask.astype(float), level=0.5)

        for contour in contours:
            ax.plot(
                contour[:, 1],
                contour[:, 0],
                color=color,
                linewidth=linewidth,
            )

    def _plot(
        slice_idx,
        zoom,
        show_ground_truth,
        show_original_segmentation,
        show_segmentation,
        show_observer_B,
        show_observer_C,
        show_observer_D,
        show_observer_E,
    ):
        z = slice_idx
        H, W = img[z].shape

        show_observers = [
            show_observer_B,
            show_observer_C,
            show_observer_D,
            show_observer_E,
        ]

        if zoom:
            fg = gt[z] | original_seg[z] | pred_seg[z]

            for obs_seg, show_obs in zip(recontours, show_observers):
                if show_obs:
                    fg |= obs_seg[z]

            if fg.any():
                ys, xs = np.where(fg)
                cy = int(np.round(np.mean(ys)))
                cx = int(np.round(np.mean(xs)))
            else:
                cy, cx = H // 2, W // 2

            h_half = int(H * zoom_fraction / 2)
            w_half = int(W * zoom_fraction / 2)

            y0 = max(0, cy - h_half)
            y1 = min(H, cy + h_half)
            x0 = max(0, cx - w_half)
            x1 = min(W, cx + w_half)

        else:
            y0, y1 = 0, H
            x0, x1 = 0, W

        img_crop = img[z][y0:y1, x0:x1]

        gt_crop = gt[z][y0:y1, x0:x1]
        original_crop = original_seg[z][y0:y1, x0:x1]
        pred_crop = pred_seg[z][y0:y1, x0:x1]

        recontour_crops = [
            r[z][y0:y1, x0:x1]
            for r in recontours
        ]

        fig, ax = plt.subplots(figsize=figsize)

        ax.imshow(img_crop, cmap="gray")

        if show_ground_truth:
            draw_contour(
                ax,
                gt_crop,
                color_gt,
                linewidth=1.8,
            )

        if show_original_segmentation:
            draw_contour(
                ax,
                original_crop,
                color_original,
                linewidth=1.8,
            )

        for i, (obs_crop, show_obs) in enumerate(
            zip(recontour_crops, show_observers)
        ):
            if show_obs:
                draw_contour(
                    ax,
                    obs_crop,
                    observer_colors[i],
                    linewidth=2.0,
                )

        if show_segmentation:
            draw_contour(
                ax,
                pred_crop,
                color_pred,
                linewidth=1.8,
            )

        ax.set_title(f"Slice {z}")
        ax.set_axis_off()

        handles = []

        if show_ground_truth:
            handles.append(
                mpatches.Patch(
                    color=color_gt,
                    label="Ground truth",
                )
            )

        if show_original_segmentation:
            handles.append(
                mpatches.Patch(
                    color=color_original,
                    label="Original segmentation",
                )
            )

        for i, (obs_name, show_obs) in enumerate(
            zip(observer_names, show_observers)
        ):
            if show_obs:
                handles.append(
                    mpatches.Patch(
                        color=observer_colors[i],
                        label=f"Observer {obs_name}",
                    )
                )

        if show_segmentation:
            handles.append(
                mpatches.Patch(
                    color=color_pred,
                    label="Generated segmentation",
                )
            )

        if handles:
            ax.legend(
                handles=handles,
                loc="upper right",
                framealpha=0.9,
            )

        plt.tight_layout()
        plt.show()

    interact(
        _plot,
        slice_idx=IntSlider(
            min=0,
            max=n_slices - 1,
            value=n_slices // 2,
            description="slice",
        ),
        zoom=Checkbox(
            value=False,
            description="zoom",
        ),
        show_ground_truth=Checkbox(
            value=True,
            description="ground truth",
        ),
        show_original_segmentation=Checkbox(
            value=True,
            description="original seg",
        ),
        show_segmentation=Checkbox(
            value=True,
            description="new seg",
        ),
        show_observer_B=Checkbox(
            value=True,
            description="observer B",
        ),
        show_observer_C=Checkbox(
            value=True,
            description="observer C",
        ),
        show_observer_D=Checkbox(
            value=True,
            description="observer D",
        ),
        show_observer_E=Checkbox(
            value=True,
            description="observer E",
        ),
    )

In [19]:
plot_recontours_interactive(
    img=data.img,
    gt=data.gt,
    original_seg=seg_handler.mask,
    pred_seg=seg_handler.predicted_seg,
    recontours=data.observer_recontours,
    observer_names=data.observer_names,
)

interactive(children=(IntSlider(value=44, description='slice', max=87), Checkbox(value=False, description='zoo…

In [23]:
# ============================================================
# Generate 3 GIFs:
# 1) Strategy 1 segmentation + prompts + observer recontour
# 2) Strategy 2 segmentation + prompts + observer recontour
# 3) Ensemble segmentation + observer recontour, no prompts
#
# Colors:
# - observer recontour: CYAN
# - new/generated delineation: YELLOW
# - dense/input mask: ORANGE
# - bbox prompt: WHITE dashed/striped line
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from skimage.measure import find_contours
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.patches import Rectangle


def _as_bool(x):
    return np.asarray(x).astype(bool)


def _get_prompt(prompt_dict, z):
    if prompt_dict is None:
        return None
    return prompt_dict.get(z, None)


def _unpack_prompt(prompt):
    if prompt is None:
        return None, None, None, None

    if isinstance(prompt, dict):
        return (
            prompt.get("points", None),
            prompt.get("point_labels", None),
            prompt.get("bbox", None),
            prompt.get("mask_input", None),
        )

    return prompt


def make_slice_loop_gif_with_observer_slices(
    img,
    pred_seg,
    observer_seg,
    dense_seg=None,
    prompt_dict=None,
    savepath="segmentation_loop.gif",
    title="Segmentation",
    fps=3,
    zoom_fraction=0.30,
    figsize=(7, 7),
    show_prompts=True,
    show_bbox=True,
):
    img = np.asarray(img)
    pred_seg = _as_bool(pred_seg)
    observer_seg = _as_bool(observer_seg)

    if dense_seg is not None:
        dense_seg = _as_bool(dense_seg)

    if img.shape != pred_seg.shape or img.shape != observer_seg.shape:
        raise ValueError(
            f"Shape mismatch: img={img.shape}, "
            f"pred_seg={pred_seg.shape}, observer_seg={observer_seg.shape}"
        )

    if dense_seg is not None and dense_seg.shape != img.shape:
        raise ValueError(f"Shape mismatch: dense_seg={dense_seg.shape}, img={img.shape}")

    z_indices = np.where(observer_seg.any(axis=(1, 2)))[0]

    if len(z_indices) == 0:
        raise ValueError("observer_seg has no foreground slices.")

    fig, ax = plt.subplots(figsize=figsize)

    def draw_contour(mask2d, color, linewidth=1.2, label=None, linestyle="-"):
        if mask2d is None or not np.any(mask2d):
            return

        contours = find_contours(mask2d.astype(float), 0.5)

        for i, c in enumerate(contours):
            ax.plot(
                c[:, 1],
                c[:, 0],
                color=color,
                linewidth=linewidth,
                linestyle=linestyle,
                label=label if i == 0 else None,
            )

    def update(frame_idx):
        ax.clear()

        z = z_indices[frame_idx]
        H, W = img[z].shape

        fg = observer_seg[z] | pred_seg[z]

        if dense_seg is not None:
            fg = fg | dense_seg[z]

        prompt = _get_prompt(prompt_dict, z)
        points, labels, bbox, mask_input = _unpack_prompt(prompt)

        if show_prompts and points is not None and len(points) > 0:
            points_arr = np.asarray(points)
            px = points_arr[:, 0]
            py = points_arr[:, 1]

            valid = (px >= 0) & (px < W) & (py >= 0) & (py < H)

            prompt_mask = np.zeros_like(fg, dtype=bool)
            prompt_mask[py[valid].astype(int), px[valid].astype(int)] = True

            fg = fg | prompt_mask

        if fg.any():
            ys, xs = np.where(fg)
            cy = int(np.round(np.mean(ys)))
            cx = int(np.round(np.mean(xs)))

            h_half = int(H * zoom_fraction / 2)
            w_half = int(W * zoom_fraction / 2)

            y0 = max(0, cy - h_half)
            y1 = min(H, cy + h_half)
            x0 = max(0, cx - w_half)
            x1 = min(W, cx + w_half)
        else:
            y0, y1 = 0, H
            x0, x1 = 0, W

        ax.imshow(img[z, y0:y1, x0:x1], cmap="gray")

        if dense_seg is not None:
            draw_contour(
                dense_seg[z, y0:y1, x0:x1],
                color="#F4A261",
                linewidth=1.0,
                label="dense/input mask",
            )

        draw_contour(
            observer_seg[z, y0:y1, x0:x1],
            color="cyan",
            linewidth=1.2,
            label="observer recontour",
        )

        draw_contour(
            pred_seg[z, y0:y1, x0:x1],
            color="yellow",
            linewidth=1.2,
            label="generated delineation",
        )

        if show_prompts and points is not None and labels is not None:
            points = np.asarray(points)
            labels = np.asarray(labels)

            pos = points[labels == 1]
            neg = points[labels == 0]

            if len(pos) > 0:
                ax.scatter(
                    pos[:, 0] - x0,
                    pos[:, 1] - y0,
                    s=20,
                    c="lightgreen",
                    marker="+",
                    linewidths=1.2,
                    label="positive prompts",
                )

            if len(neg) > 0:
                ax.scatter(
                    neg[:, 0] - x0,
                    neg[:, 1] - y0,
                    s=20,
                    c="lightcoral",
                    marker="x",
                    linewidths=1.2,
                    label="negative prompts",
                )

        if show_prompts and show_bbox and bbox is not None:
            boxes = np.asarray(bbox)

            if boxes.ndim == 1:
                boxes = boxes[None, :]

            for i, (bx0, by0, bx1, by1) in enumerate(boxes):
                rect = Rectangle(
                    (bx0 - x0, by0 - y0),
                    bx1 - bx0,
                    by1 - by0,
                    fill=False,
                    edgecolor="white",
                    linewidth=1.0,
                    linestyle=(0, (4, 3)),
                    label="bbox prompt" if i == 0 else None,
                )
                ax.add_patch(rect)

        ax.set_title(
            f"{title} | observer slice {frame_idx + 1}/{len(z_indices)} | z={z}"
        )
        ax.set_axis_off()

        handles, labels_ = ax.get_legend_handles_labels()
        unique = dict(zip(labels_, handles))

        if len(unique) > 0:
            ax.legend(
                unique.values(),
                unique.keys(),
                loc="upper right",
                framealpha=0.85,
            )

    ani = FuncAnimation(
        fig,
        update,
        frames=len(z_indices),
        interval=1000 / fps,
        repeat=True,
    )

    savepath = Path(savepath)
    savepath.parent.mkdir(parents=True, exist_ok=True)

    ani.save(savepath, writer=PillowWriter(fps=fps))
    plt.close(fig)

    print(f"Saved GIF to: {savepath}")


def make_three_strategy_gifs_observer_slices_only(
    img,
    strategy_segmentations,
    strategy_prompts,
    ensemble_segmentation,
    observer_seg,
    dense_seg=None,
    output_folder="slice_gifs_observer_slices_only",
    fps=3,
    zoom_fraction=0.30,
    figsize=(7, 7),
):
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    for strategy_name, pred_seg in strategy_segmentations.items():

        prompt_dict = strategy_prompts.get(strategy_name, None)

        make_slice_loop_gif_with_observer_slices(
            img=img,
            pred_seg=pred_seg,
            observer_seg=observer_seg,
            dense_seg=dense_seg,
            prompt_dict=prompt_dict,
            savepath=output_folder / f"{strategy_name}_with_prompts_and_observer.gif",
            title=strategy_name,
            fps=fps,
            zoom_fraction=zoom_fraction,
            figsize=figsize,
            show_prompts=True,
            show_bbox=True,
        )

    make_slice_loop_gif_with_observer_slices(
        img=img,
        pred_seg=np.zeros((88, 1024, 1024), dtype=np.float32),
        observer_seg=observer_seg,
        dense_seg=dense_seg,
        prompt_dict=None,
        savepath=output_folder / "ensemble_with_observer_no_prompts.gif",
        title="Ensemble segmentation",
        fps=fps,
        zoom_fraction=zoom_fraction,
        figsize=figsize,
        show_prompts=False,
        show_bbox=False,
    )


# ============================================================
# EXAMPLE CALL
# ============================================================

strategy_segmentations = {
    "Dense_and_nietjes": seg_handler.segs_per_set["Dense_and_nietjes"],
    "Uncertainty_bboxes": seg_handler.segs_per_set["Uncertainty_bboxes"],
    "All": seg_handler.segs_per_set["All"],
}

strategy_prompts = {
    "Dense_and_nietjes": combine_prompt_sets(
        prompt_dict_list=[dense_prompt, nietjes_prompts]
    ),
    "Uncertainty_bboxes": bbox_prompts,
    "All": all,
}

make_three_strategy_gifs_observer_slices_only(
    img=data.img,
    strategy_segmentations=strategy_segmentations,
    strategy_prompts=strategy_prompts,
    ensemble_segmentation=seg_handler.predicted_seg,
    observer_seg=data.observer_recontours[3],
    dense_seg=seg_handler.mask,
    output_folder="test",
    fps=1,
    zoom_fraction=0.30,
    figsize=(7, 7),
)

Saved GIF to: test\Dense_and_nietjes_with_prompts_and_observer.gif
Saved GIF to: test\Uncertainty_bboxes_with_prompts_and_observer.gif
Saved GIF to: test\All_with_prompts_and_observer.gif
Saved GIF to: test\ensemble_with_observer_no_prompts.gif
